# Prototipo: descripción clínica de una fotografía de fondo de ojo

Prototipo del trabajo de graduación sobre *surgical fine-tuning* de MedGemma (`google/medgemma-1.5-4b-it`), con foco en glaucoma. Toda la lógica está en `prototype/prototipo.py`; este cuaderno solo carga, muestra la interfaz y ejecuta un ejemplo.

**Aviso.** No es un sistema clínico. La fidelidad del texto es a los campos (vCDR, hCDR, decisión), no a la imagen.

Dos modos: **(a)** una imagen de la validación de Chákṣu, con la referencia de los expertos tras el resultado; **(b)** una imagen propia sin referencia. En ambos se elige el adaptador entre los mejores del estudio. Pasos: ejecutar la celda 1 (carga, la más lenta), la celda 2 (interfaz) y usar la interfaz.

## 1. Carga
MedGemma se carga una vez en bf16 y se le montan los cuatro adaptadores de medición y el de redacción; `set_adapter` elige cuál actúa.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "prototype" else Path.cwd()
assert (REPO / "scripts").is_dir(), f"{REPO} no es la raíz del repositorio"
sys.path.insert(0, str(REPO))

from IPython.display import HTML, display

from prototype.prototipo import Interface, OPTIONS, infer, load_prototype

SEED = 42  # semilla de entrenamiento de los adaptadores (42, 43 o 44)
proto = load_prototype(SEED)
print("adaptadores montados:", sorted(proto.model.peft_config))

## 2. Interfaz

In [ ]:
ui = Interface(proto)
display(ui.root)

## 3. Ejemplo y verificación

Misma ruta que los botones, llamada desde código: sirve para comprobar de punta a punta sin navegador (`slurm/notebook_template.slurm`). Se usan imágenes de la validación de Chákṣu; en el modo (b) una de ellas se trata como desconocida (se «sube» como archivo) y su resultado no muestra referencia.

In [ ]:
from datetime import datetime

ids = dict((label, image_id) for image_id, label in ui.examples)
show = lambda widget: display(HTML(widget.value))

# Modo (a): imagen con referencia conocida (una sospechosa y una normal), adaptador A all-linear.
ui.option.value = "A_all_linear"
for label in ("Bosch · sospechoso", "Remidio · normal"):
    ui.example.value = ids[label]
    assert ui.image_id.value == ids[label]
    ui.run_reference()
    display(HTML(f"<h4>Modo (a): {label}</h4>"))
    show(ui.out_ref)
    assert "Referencia de los expertos" in ui.out_ref.value

# Modo (b): imagen propia sin referencia. Se sube como archivo, sin pasar por la referencia de Chákṣu.
path = REPO.parent / "datasets" / ui.validation[ids["Forus · sospechoso"]]["image"]
ui.upload.value = ({"name": path.name, "type": "image/png", "size": path.stat().st_size,
                   "last_modified": datetime.fromtimestamp(path.stat().st_mtime), "content": path.read_bytes()},)
ui.run_own()
display(HTML("<h4>Modo (b): imagen propia, sin referencia</h4>"))
show(ui.out_own)
assert "Referencia" not in ui.out_own.value and "expertos" not in ui.out_own.value

# Modo (b) con anotaciones de un oftalmólogo y una referencia propia para comparar.
ui.use_annotations.value, ui.use_reference.value = True, True
for box, values in ((ui.ann, (0.60, 0.58, "GLAUCOMA SUSPECT")), (ui.ref, (0.55, 0.50, "GLAUCOMA SUSPECT"))):
    box.vcdr.value, box.hcdr.value, box.decision.value = values
ui.run_own()
display(HTML("<h4>Modo (b) con anotaciones y referencia propia</h4>"))
show(ui.out_own)

# El selector cambia de verdad de adaptador: se recorre cada opción, se comprueba el adaptador activo y que las salidas
# crudas de la etapa 1 de los adaptadores A no son todas iguales.
ui.use_annotations.value = ui.use_reference.value = False
image = ui._uploaded()[1]
raws = {}
for key in OPTIONS:
    ui.option.value = key
    result = infer(proto, image, key)
    assert proto.model.active_adapters == ([key] if OPTIONS[key]["path"] == "B" else ["verbalize"]), (key, proto.model.active_adapters)
    raws[key] = result["adapter"]["raw_stage1"] or result["adapter"]["text"]
    print(f"{key}: {raws[key]!r}")
assert len(set(raws.values())) > 1, "el selector no cambió la salida"
print("selector verificado:", sorted(raws))